# Setup: three main tasks.
1) *agent_from_lamp*: transform individuals from survey data into agents. We use the *Person* struct and constructor.
2) *add_agents!*: add agents to their respective cohorts.
3) *setup_sim*: get a simulation object with a population of agents based on empirical data. 

## 1. Transform individuals to agents.
- *agent_from_lamp*: transform individuals to agents. 
- Individuals' characteristics from survey data are transformed to agent's property *values*. 

In [2]:
function agent_from_lamp(person_years)
	agent = Person(birth_year(person_years))
	agent.values = lamp_to_values(person_years)
	agent
end

agent_from_lamp (generic function with 1 method)

## 2. Add agents to their respective cohorts.
Use weights if available. In this case, push (add) as many agents as the weight from lamp into pop.

In [6]:
function add_agents!(cohort_pop, pop; use_weights = true)
	if length(cohort_pop) == 0
		return
	end

	for person_years in cohort_pop
		w = use_weights ? weight(person_years) : 1
		# Push as many agents as the weight from lamp into pop. 
		for j in 1:w
			push!(pop, agent_from_lamp(person_years))
		end
	end
end

add_agents! (generic function with 1 method)

In [ ]:
# Add agents function
function add_random_agents!(cohort_pop, pop, number; use_weights = true)
	if length(cohort_pop) == 0
		return
	end

	count = 0
	# add agents until we have enough
	while count < number
		i = rand(length(cohort_pop))
		w = use_weights ? weight(cohort_pop[i]) : 1
		w = min(w, number - count) # we don't want to overshoot
		for j in 1:w
			push!(pop, agent_from_lamp(cohort_pop[i]))
			count += 1
		end
	end
end

## 3. The  *setup_sim* function:
- Generate *cohort*: the number of people in each cohort.
- Generate *cohort_pop*: the actual individuals within each cohort.
- **Populate *cohort_pop* using survey data.**
- Output: *sim* object with pop, raw data, cogort size, and pop_cohort. 

In [8]:
function setup_sim(data, seed; use_weights = true)
	# find out which type of data we are working on
	DType = valtype(data)

	# number of individuals entering the population per year
	cohort = zeros(Int, 130) # Vector with zeros.

	# actual data by year
	# this could be abbreviated as: cohort_pop = [ Vector{DType}() for i in 1:130 ] 
	cohort_pop = Vector{Vector{DType}}() # Vector with empty vectors.
	for i in 1:130 # to account for all agents and their person-years
		push!(cohort_pop, Vector{DType}()) # Manually defining cohort_pop
	end

	# person_years has type DType (which is actually Vector{DataFrameRow})
	for (person_id, person_years) in data
		# year we are dealing with
		index = floor(Int, birth_year(person_years)) - 1929 # 1929 is hardcoded! Change?
		# count
		cohort[index] = cohort[index] + (use_weights ? weight(person_years[1]) : 1)
		# and add to data by "birth" year
		push!(cohort_pop[index], person_years)
	end

	Random.seed!(seed)
	sim = Simulation(Person[], data, cohort, cohort_pop) # remember that last statement is the returned value.
end

setup_sim (generic function with 1 method)